In [ ]:
import warnings
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("future.no_silent_downcasting", True)

from pathlib import Path
import numpy as np
import pandas as pd
import joblib

MODEL_PATH = "/kaggle/input/models/jek1wantaufik/buddy/scikitlearn/spaceship/1/output/model.pkl"

TEST_PATH = "/kaggle/input/competitions/spaceship-titanic/test.csv"
SUB_PATH = "/kaggle/input/competitions/spaceship-titanic/sample_submission.csv"

CFG = {
    "spend_cols": ["RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck"]
}

def feature_engineering(df):
    df = df.copy()

    cabin = df["Cabin"].fillna("U/0/U").str.split("/", expand=True)
    df["CabinDeck"] = cabin[0]
    df["CabinNum"] = pd.to_numeric(cabin[1], errors="coerce")
    df["CabinSide"] = cabin[2]

    name = df["Name"].fillna("X X").str.split(" ", n=1, expand=True)
    df["Surname"] = name[1]

    df["GroupId"] = df["PassengerId"].str.split("_").str[0].astype(int)

    for c in CFG["spend_cols"]:
        df[c] = df[c].fillna(0)

    df["TotalSpend"] = df[CFG["spend_cols"]].sum(axis=1)
    df["NoSpend"] = (df["TotalSpend"] == 0).astype(int)

    df["CryoSleep"] = df["CryoSleep"].fillna(False).astype(str)
    df["VIP"] = df["VIP"].fillna(False).astype(str)

    df = df.drop(columns=["Cabin", "Name", "PassengerId"], errors="ignore")

    return df

model = joblib.load(MODEL_PATH)

test = pd.read_csv(TEST_PATH)
sub = pd.read_csv(SUB_PATH)

X_test = feature_engineering(test)

cat_cols = X_test.select_dtypes(include=["object"]).columns.tolist()

for c in cat_cols:
    X_test[c] = X_test[c].astype(str)

preds = model.predict(X_test)

sub["Transported"] = preds.astype(bool)

sub.to_csv("/kaggle/working/submission.csv", index=False)

print("Submission created")